# Graph Visualization

This notebook visualizes the message-passing graph used by neural-lam's GNN.
It shows grid nodes, mesh nodes, and the three edge types: G2M, M2G, and M2M.

**Prerequisites:** Complete the Hello World notebook (Section 3 — Graph Generation).

> **Note:** Run from the repository root, or from `docs/notebooks/` — the setup cell handles the working directory automatically.

## 1. Setup

In [ ]:
%matplotlib inline
import os

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch

# Fix random seed for reproducible edge sampling
np.random.seed(42)

# Resolve repo root
while not os.path.isdir(os.path.join(os.getcwd(), "neural_lam")):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        raise RuntimeError("Could not find repo root (neural_lam/ not found)")
    os.chdir(parent)
print("Working directory:", os.getcwd())

## 2. Load Graph Tensors

neural-lam stores graph edges as PyTorch `.pt` files under `<datastore_root>/graph/<name>/`.

**Important:** `m2m_edge_index.pt`, `m2m_features.pt`, and `mesh_features.pt` are always saved as **Python lists** of tensors (one entry per mesh level). Even for the single-level graph (`L=1`) they are lists. We unwrap level 0 with `[0]`.

`g2m_edge_index.pt` and `m2g_edge_index.pt` are plain tensors.

In [ ]:
GRAPH_DIR = "tests/datastore_examples/mdp/danra_100m_winds/graph/1level"

if not os.path.exists(GRAPH_DIR):
    raise FileNotFoundError(
        f"Graph directory not found at {GRAPH_DIR}.\n"
        "Run the Hello World notebook (Section 3 — Graph Generation) first."
    )


def load_pt(name):
    path = os.path.join(GRAPH_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing graph file: {path}")
    return torch.load(path, weights_only=True)


# g2m and m2g are plain tensors — shape (2, E)
g2m_edge_index = load_pt("g2m_edge_index.pt")  # tensor (2, E_g2m)
m2g_edge_index = load_pt("m2g_edge_index.pt")  # tensor (2, E_m2g)

# m2m, m2m_features and mesh_features are LISTS of tensors (one per mesh level).
# For a 1-level graph the list has one entry — we take index [0].
m2m_edge_index = load_pt("m2m_edge_index.pt")[0]  # tensor (2, E_m2m)
m2m_features   = load_pt("m2m_features.pt")[0]    # tensor (E_m2m, d_features)
mesh_features  = load_pt("mesh_features.pt")[0]   # tensor (N_mesh, 2)

print("✅ Graph loaded")
print(f"   Grid→Mesh edges  (G2M): {g2m_edge_index.shape[1]:,}")
print(f"   Mesh→Grid edges  (M2G): {m2g_edge_index.shape[1]:,}")
print(f"   Mesh→Mesh edges  (M2M): {m2m_edge_index.shape[1]:,}")
print(f"   Mesh nodes             : {mesh_features.shape[0]:,}")

## 3. Load Grid Positions

Grid coordinates come from the datastore via `get_xy`, which returns projected (x, y) positions. We normalise to [-1, 1] for consistent plotting.

In [ ]:
CONFIG_PATH = "tests/datastore_examples/mdp/danra_100m_winds/config.yaml"

from neural_lam.config import load_config_and_datastore

_, datastore = load_config_and_datastore(config_path=CONFIG_PATH)

# get_xy returns (N_grid, 2) array of projected coordinates
xy      = datastore.get_xy("state", stacked=True)
pos_max = np.abs(xy).max()
grid_x  = xy[:, 0] / pos_max
grid_y  = xy[:, 1] / pos_max
n_grid  = len(grid_x)
n_mesh  = mesh_features.shape[0]

mesh_x = mesh_features[:, 0].numpy()
mesh_y = mesh_features[:, 1].numpy()

# Grid shape (no square-root assumptions)
grid_shape = datastore.grid_shape_state
print(f"   Grid nodes : {n_grid:,}  ({grid_shape.y}×{grid_shape.x} grid)")
print(f"   Mesh nodes : {n_mesh:,}")

## 4. Full Graph Overview

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

# ── Runtime-safe G2M index decoding ──────────────────────────────────────────
# neural-lam may store g2m_edge_index using either:
#   (a) global node space: grid and mesh share a contiguous index range
#       (grid-first or mesh-first ordering)
#   (b) local per-type space: row0=grid(0..n_grid-1), row1=mesh(0..n_mesh-1)
# We auto-detect at runtime by inspecting which row spans which numeric range.
# ─────────────────────────────────────────────────────────────────────────────

row0 = g2m_edge_index[0].numpy()   # shape (E_g2m,)
row1 = g2m_edge_index[1].numpy()   # shape (E_g2m,)


def matches_range(values, offset, length):
    return values.min() >= offset and values.max() < offset + length


def decode_g2m_rows(row_a, row_b):
    for row_a_is_grid in (True, False):
        grid_row, mesh_row = (row_a, row_b) if row_a_is_grid else (row_b, row_a)
        grid_offset = next(
            (off for off in (0, n_mesh) if matches_range(grid_row, off, n_grid)),
            None,
        )
        mesh_offset = next(
            (off for off in (0, n_grid) if matches_range(mesh_row, off, n_mesh)),
            None,
        )
        if grid_offset is not None and mesh_offset is not None:
            return row_a_is_grid, grid_offset, mesh_offset
    raise ValueError(
        "Could not infer g2m_edge_index convention (unexpected index ranges)"
    )


row0_is_grid, grid_offset, mesh_offset = decode_g2m_rows(row0, row1)
if row0_is_grid:
    grid_row, mesh_row = row0, row1
    grid_label, mesh_label = "row0", "row1"
else:
    grid_row, mesh_row = row1, row0
    grid_label, mesh_label = "row1", "row0"


def describe(label, is_grid, offset):
    if offset == 0:
        scope = "local"
    elif is_grid and offset == n_mesh:
        scope = "global, mesh-first offset"
    elif (not is_grid) and offset == n_grid:
        scope = "global, grid-first offset"
    else:
        scope = f"offset={offset}"
    role = "grid" if is_grid else "mesh"
    return f"{label}={role}({scope})"


grid_idx = (grid_row - grid_offset).astype(int)
mesh_idx = (mesh_row - mesh_offset).astype(int)
g2m_grid_xy = np.stack([grid_x[grid_idx], grid_y[grid_idx]], axis=1)
g2m_mesh_xy = np.stack([mesh_x[mesh_idx], mesh_y[mesh_idx]], axis=1)

print(
    "Convention detected: "
    + "; ".join(
        [
            describe(grid_label, True, grid_offset),
            describe(mesh_label, False, mesh_offset),
        ]
    )
)

# Sample and plot G2M edges
n_sample = min(2000, len(grid_idx))
idx_sample = np.random.choice(len(grid_idx), size=n_sample, replace=False)
for i in idx_sample:
    ax.plot(
        [g2m_grid_xy[i, 0], g2m_mesh_xy[i, 0]],
        [g2m_grid_xy[i, 1], g2m_mesh_xy[i, 1]],
        color="orange",
        alpha=0.05,
        linewidth=0.4,
    )

# M2M edges — m2m always uses local mesh indices (0..n_mesh-1)
m2m_src = m2m_edge_index[0].numpy()
m2m_dst = m2m_edge_index[1].numpy()
for i in range(len(m2m_src)):
    ax.plot(
        [mesh_x[m2m_src[i]], mesh_x[m2m_dst[i]]],
        [mesh_y[m2m_src[i]], mesh_y[m2m_dst[i]]],
        color="royalblue",
        alpha=0.5,
        linewidth=0.6,
    )

ax.scatter(grid_x, grid_y, s=1, c="black", alpha=0.3)
ax.scatter(mesh_x, mesh_y, s=20, c="royalblue", zorder=5)

legend_handles = [
    mpatches.Patch(
        color="orange",
        label=f"G2M edges ({g2m_edge_index.shape[1]:,}, sampled)",
    ),
    mpatches.Patch(color="royalblue", label=f"M2M edges ({m2m_edge_index.shape[1]:,})"),
    mpatches.Patch(color="black", label=f"Grid nodes ({n_grid:,})"),
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=9)
ax.set_title("neural-lam Graph — 1-Level (L1-LAM)", fontsize=13)
ax.set_xlabel("Normalised X")
ax.set_ylabel("Normalised Y")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()



## 5. Mesh Node Degree Distribution

How many M2M edges connect to each mesh node?

In [ ]:
degrees = np.bincount(m2m_edge_index[1].numpy(), minlength=n_mesh)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc = axes[0].scatter(mesh_x, mesh_y, c=degrees, cmap="plasma", s=40, zorder=5)
plt.colorbar(sc, ax=axes[0], label="Node degree (M2M)")
axes[0].set_title("Mesh Node Degree")
axes[0].set_aspect("equal")

axes[1].hist(degrees, bins=20, color="royalblue", edgecolor="white")
axes[1].set_xlabel("Degree")
axes[1].set_ylabel("Count")
axes[1].set_title("Degree Distribution")

plt.tight_layout()
plt.show()

print(f"Mean degree : {degrees.mean():.2f}")
print(f"Max degree  : {degrees.max()}")
print(f"Min degree  : {degrees.min()}")

## 6. Edge Count Summary

In [ ]:
total_edges = (
    g2m_edge_index.shape[1]
    + m2g_edge_index.shape[1]
    + m2m_edge_index.shape[1]
)

labels = ["G2M\n(Grid→Mesh)", "M2G\n(Mesh→Grid)", "M2M\n(Mesh→Mesh)"]
counts = [
    g2m_edge_index.shape[1],
    m2g_edge_index.shape[1],
    m2m_edge_index.shape[1],
]
colors = ["orange", "green", "royalblue"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, counts, color=colors, edgecolor="white", width=0.5)
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 100,
        f"{count:,}",
        ha="center", va="bottom", fontsize=10,
    )
ax.set_ylabel("Number of edges")
ax.set_title(f"Edge Count by Type  (total: {total_edges:,})")
plt.tight_layout()
plt.show()